In [ ]:
# Core imports
import os
import re
import cv2
import io
import base64
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Jupyter widgets
from IPython.display import HTML, display
import ipywidgets as widgets

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# Scikit-image
from skimage.measure import label as connected_components
from skimage.measure import regionprops

# Progress bar
from tqdm import tqdm

from torch.utils.data import WeightedRandomSampler

In [ ]:
# Install dependencies
!pip install scikit-image lion-pytorch ranger21 -q
# Optimizers
from lion_pytorch import Lion
from ranger21 import Ranger21

# Data Loading

In [ ]:
def center_pad_to_square_img(img_bgr, pad_color=(255,255,255)):
    h, w = img_bgr.shape[:2]
    side = max(h, w)
    top = (side - h) // 2
    bottom = side - h - top
    left = (side - w) // 2
    right = side - w - left
    return cv2.copyMakeBorder(img_bgr, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_color)

def center_pad_to_square_mask(mask_gray, pad_value=0):
    h, w = mask_gray.shape[:2]
    side = max(h, w)
    top = (side - h) // 2
    bottom = side - h - top
    left = (side - w) // 2
    right = side - w - left
    return cv2.copyMakeBorder(mask_gray, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_value)

def load_img_mask_pairs(folder: str, folder_mask, target_size=(512, 512)):
    """Load full resolution images and masks"""
    id_re = re.compile(r'^(img_|mask_)(\d+)\.png$', re.IGNORECASE)
    img_map, mask_map = {}, {}

    print("Loading masks....")
    for name in os.listdir(folder_mask):
        m = id_re.match(name)
        if not m:
            continue
        prefix, idx = m.group(1), m.group(2)
        if prefix.lower().startswith("mask_"):
            mask_map[idx] = os.path.join(folder_mask, name)
        else:
            img_map[idx] = os.path.join(folder, name)

    pairs = []

    for idx in sorted(img_map.keys(), key=lambda x: int(x)):
        img = cv2.imread(img_map[idx], cv2.IMREAD_COLOR)
        mask = cv2.imread(mask_map[idx], cv2.IMREAD_GRAYSCALE)

        # Binary Mask
        mask_bin = (mask > 127).astype(np.uint8)
        pairs.append({
            "id": idx,
            "img": img,
            "mask": mask_bin,
            "img_path": img_map[idx],
            "mask_path": mask_map[idx],
        })

    return pairs

In [ ]:
# Connect to GoogleDrive
from google.colab import drive
drive.mount('/gdrive')

INPUT_DIRECTORY = '/gdrive/MyDrive/AN2DL_Chall2/AN2DL_Clean_DS/'
train_labels = pd.read_csv(INPUT_DIRECTORY + 'train_labels.csv')
train_data = load_img_mask_pairs(INPUT_DIRECTORY + 'train_data_normalized',INPUT_DIRECTORY + 'train_data_normalized' , target_size=(512, 512))
test_data = load_img_mask_pairs(INPUT_DIRECTORY + 'test_data_normalized',INPUT_DIRECTORY + 'test_data_normalized' , target_size=(512, 512))
print(f"Loaded {len(train_data)} train images")
print(f"Loaded {len(test_data)} test images")

In [ ]:
CLASS_NAMES = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']
NAME2IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
IDX2NAME = {i: n for i, n in enumerate(CLASS_NAMES)}

def to_idx(label):
    return NAME2IDX[label] if isinstance(label, str) else int(label)

def to_name(idx):
    return IDX2NAME[int(idx)]

# Data Exploration

In [ ]:
train_labels['label'].value_counts().plot(kind='pie', subplots=True, autopct='%1.0f%%')

In [ ]:
def to_b64_rgb(arr_uint8_hwc):
    im = Image.fromarray(arr_uint8_hwc)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def prep_img_thumb(img_float_rgb, size=192):
    img = (np.clip(img_float_rgb, 0, 1) * 255).astype(np.uint8)
    thumb = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return to_b64_rgb(thumb)

def prep_mask_thumb(mask_uint8_hw1, size=192, colorize=False):
    m = mask_uint8_hw1[..., 0]
    if colorize:
        rgb = np.zeros((m.shape[0], m.shape[1], 3), dtype=np.uint8)
        rgb[..., 2] = (m > 0).astype(np.uint8) * 255
    else:
        gray = (m > 0).astype(np.uint8) * 255
        rgb = np.repeat(gray[..., None], 3, axis=2)
    thumb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_NEAREST)
    return to_b64_rgb(thumb)

def prep_overlay_thumb(img_float_rgb, mask_uint8_hw1, size=192,
                       color=(255, 0, 0), alpha=0.45, outline=True, fill=False):
    img = (np.clip(img_float_rgb, 0, 1) * 255).astype(np.uint8)
    m = mask_uint8_hw1[..., 0].astype(np.uint8)
    overlay = img.copy()

    mask_bool = m > 0
    if fill and mask_bool.any():
        color_arr = np.zeros_like(img, dtype=np.uint8); color_arr[:] = np.array(color, dtype=np.uint8)
        overlay[mask_bool] = (overlay[mask_bool].astype(np.float32) * (1 - alpha) +
                              color_arr[mask_bool].astype(np.float32) * alpha).astype(np.uint8)

    if outline and mask_bool.any():
        kernel = np.ones((3,3), np.uint8)
        dil = cv2.dilate(m, kernel, iterations=1)
        er  = cv2.erode(m, kernel, iterations=1)
        edge = (dil > 0) & (er == 0)
        overlay[edge] = (255, 255, 255)

    thumb = cv2.resize(overlay, (size, size), interpolation=cv2.INTER_AREA)
    return to_b64_rgb(thumb)

def build_gallery_side_by_side(pairs, page=1, per_page=100, thumb_size=192, colorize_mask=False):
    start = (page - 1) * per_page
    chunk = pairs[start:start + per_page]
    cards = []
    for p in chunk:
        b64_img = prep_img_thumb(p["img"], size=thumb_size)
        b64_msk = prep_mask_thumb(p["mask"], size=thumb_size, colorize=colorize_mask)
        cards.append(f"""
        <div class=\"card\">
          <div class=\"pair\">
            <div class=\"pane\">
              <img src=\"data:image/jpeg;base64,{b64_img}\" loading=\"lazy\">
              <div class=\"meta\">img id {p['id']}</div>
            </div>
            <div class=\"pane\">
              <img src=\"data:image/jpeg;base64,{b64_msk}\" loading=\"lazy\">
              <div class=\"meta\">mask id {p['id']}</div>
            </div>
          </div>
        </div>
        """)
    total_pages = (len(pairs) + per_page - 1) // per_page
    html = f"""
    <style>
      .grid {{
        display: grid;
        grid-template-columns: repeat(auto-fill, minmax({thumb_size*2 + 24}px, 1fr));
        gap: 10px;
      }}
      .card {{
        border: 1px solid #ddd; border-radius: 6px; background: #fff; overflow: hidden;
        padding: 8px;
      }}
      .pair {{ display: flex; gap: 8px; align-items: center; justify-content: center; }}
      .pane {{ text-align: center; }}
      .pane img {{ width: {thumb_size}px; height: {thumb_size}px; object-fit: cover; display: block; }}
      .meta {{ font-size: 12px; color: #555; margin-top: 4px; }}
    </style>
    <div>Page {page} / {total_pages}</div>
    <div class=\"grid\">{''.join(cards)}</div>
    """
    return HTML(html)

def build_gallery_overlay(pairs, page=1, per_page=100, thumb_size=192, color=(255,0,0), alpha=0.45, outline=True):
    start = (page - 1) * per_page
    chunk = pairs[start:start + per_page]
    cards = []
    for p in chunk:
        b64_overlay = prep_overlay_thumb(p["img"], p["mask"], size=thumb_size, color=color, alpha=alpha, outline=outline)
        cards.append(f"""
        <div class=\"card\">
          <div class=\"pane\">
            <img src=\"data:image/jpeg;base64,{b64_overlay}\" loading=\"lazy\">
            <div class=\"meta\">id {p['id']} — overlay</div>
          </div>
        </div>
        """)
    total_pages = (len(pairs) + per_page - 1) // per_page
    html = f"""
    <style>
      .grid {{
        display: grid;
        grid-template-columns: repeat(auto-fill, minmax({thumb_size + 24}px, 1fr));
        gap: 10px;
      }}
      .card {{
        border: 1px solid #ddd; border-radius: 6px; background: #fff; overflow: hidden;
        padding: 8px;
      }}
      .pane {{ text-align: center; }}
      .pane img {{ width: {thumb_size}px; height: {thumb_size}px; object-fit: cover; display: block; }}
      .meta {{ font-size: 12px; color: #555; margin-top: 4px; }}
    </style>
    <div>Page {page} / {total_pages}</div>
    <div class=\"grid\">{''.join(cards)}</div>
    """
    return HTML(html)

def show_gallery(pairs, per_page=100, thumb_size=192, colorize_mask=False):
    total_pages = max(1, (len(pairs) + per_page - 1) // per_page)
    state = {"page": 1}
    out = widgets.Output()
    btn_prev = widgets.Button(description="Prev", button_style="")
    btn_next = widgets.Button(description="Next", button_style="primary")

    def render():
        with out:
            out.clear_output(wait=True)
            display(build_gallery_side_by_side(
                pairs, page=state["page"], per_page=per_page,
                thumb_size=thumb_size, colorize_mask=colorize_mask
            ))

    def on_prev(_):
        if state["page"] > 1:
            state["page"] -= 1
            render()

    def on_next(_):
        if state["page"] < total_pages:
            state["page"] += 1
            render()

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    render()
    display(widgets.HBox([btn_prev, btn_next]), out)

def show_gallery_overlay(pairs, per_page=100, thumb_size=192, color=(255,0,0), alpha=0.45, outline=True):
    total_pages = max(1, (len(pairs) + per_page - 1) // per_page)
    state = {"page": 1}
    out = widgets.Output()
    btn_prev = widgets.Button(description="Prev", button_style="")
    btn_next = widgets.Button(description="Next", button_style="primary")

    def render():
        with out:
            out.clear_output(wait=True)
            display(build_gallery_overlay(
                pairs, page=state["page"], per_page=per_page,
                thumb_size=thumb_size, color=color, alpha=alpha, outline=outline
            ))

    def on_prev(_):
        if state["page"] > 1:
            state["page"] -= 1
            render()

    def on_next(_):
        if state["page"] < total_pages:
            state["page"] += 1
            render()

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    render()
    display(widgets.HBox([btn_prev, btn_next]), out)

In [ ]:
display(show_gallery(train_data, per_page=100, thumb_size=192, colorize_mask=False))

# Multi-Crop Strategy

Extract multiple crops from disconnected mask regions in the same image

In [ ]:
def extract_region_crops(img_hwc, mask_hw, min_area=100, padding=20, target_size=(224, 224)):
    """
    Extract crop for each connected region in the mask.
    Returns: list of dicts with keys: img, mask, bbox, area
    """
    labeled_mask = connected_components(mask_hw, connectivity=2)
    regions = regionprops(labeled_mask)

    crops = []
    h, w = img_hwc.shape[:2]

    for region in regions:
        if region.area < min_area:
            continue

        min_row, min_col, max_row, max_col = region.bbox
        min_row = max(0, min_row - padding)
        min_col = max(0, min_col - padding)
        max_row = min(h, max_row + padding)
        max_col = min(w, max_col + padding)

        crop_img = img_hwc[min_row:max_row, min_col:max_col]
        crop_mask = mask_hw[min_row:max_row, min_col:max_col]

        crop_img_resized = cv2.resize(crop_img, target_size, interpolation=cv2.INTER_AREA)
        crop_mask_resized = cv2.resize(crop_mask.astype(np.uint8), target_size, interpolation=cv2.INTER_NEAREST)

        crops.append({
            'img': crop_img_resized,
            'mask': crop_mask_resized,
            'bbox': (min_row, min_col, max_row, max_col),
            'area': region.area
        })

    if len(crops) == 0:
        crop_img_resized = cv2.resize(img_hwc, target_size, interpolation=cv2.INTER_AREA)
        crop_mask_resized = cv2.resize(mask_hw.astype(np.uint8), target_size, interpolation=cv2.INTER_NEAREST)
        crops.append({
            'img': crop_img_resized,
            'mask': crop_mask_resized,
            'bbox': (0, 0, h, w),
            'area': mask_hw.sum()
        })

    return crops

print("📸 Visualizing multi-crop strategy...")
sample_pair = train_data[0]
crops = extract_region_crops(sample_pair['img'], sample_pair['mask'], min_area=0, padding=30)

print(f"Found {len(crops)} regions in image {sample_pair['id']}")

fig, axes = plt.subplots(2, len(crops) + 1, figsize=(4 * (len(crops) + 1), 8))

axes[0, 0].imshow(sample_pair['img'])
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

axes[1, 0].imshow(sample_pair['mask'], cmap='gray')
axes[1, 0].set_title('Full Mask')
axes[1, 0].axis('off')

for i, crop in enumerate(crops):
    axes[0, i+1].imshow(crop['img'])
    axes[0, i+1].set_title(f'Crop {i+1} (area={crop["area"]})')
    axes[0, i+1].axis('off')

    axes[1, i+1].imshow(crop['mask'], cmap='gray')
    axes[1, i+1].set_title(f'Mask {i+1}')
    axes[1, i+1].axis('off')

plt.tight_layout()
plt.show()

# Spatial Attention Mechanism

Use mask as attention gradient for feature weighting

In [ ]:
class SpatialAttention(nn.Module):
    """Spatial attention guided by the mask."""
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features, mask_guidance):
        attention_map = self.conv(features)
        attention_map = self.sigmoid(attention_map)
        guided_attention = attention_map * mask_guidance
        guided_attention = guided_attention / (guided_attention.sum(dim=(2, 3), keepdim=True) + 1e-8)
        attended_features = features * guided_attention
        return attended_features, guided_attention

print("✅ SpatialAttention module defined")

In [ ]:
class CBAM_SoftMask_Attention(nn.Module):
    """CBAM attention with soft mask guidance to keep context."""
    def __init__(self, in_channels, reduction=16, leak_weight=0.15):
        super().__init__()
        self.leak_weight = leak_weight

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.channel_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )

        self.spatial_conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm2d(1)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, features, mask_guidance):
        batch_size, channels, height, width = features.size()

        avg_out = self.channel_mlp(self.avg_pool(features))
        max_out = self.channel_mlp(self.max_pool(features))
        channel_att = self.sigmoid(avg_out + max_out)
        features_refined = features * channel_att

        avg_spatial = torch.mean(features_refined, dim=1, keepdim=True)
        max_spatial, _ = torch.max(features_refined, dim=1, keepdim=True)
        spatial_input = torch.cat([avg_spatial, max_spatial], dim=1)
        spatial_att = self.sigmoid(self.spatial_conv(spatial_input))

        soft_mask = mask_guidance * (1.0 - self.leak_weight) + self.leak_weight
        guided_attention = spatial_att * soft_mask
        guided_attention = guided_attention / (guided_attention.sum(dim=(2, 3), keepdim=True) + 1e-8)
        attended_features = features_refined * guided_attention * height * width
        return attended_features, guided_attention

print("✅ CBAM + SoftMask Attention defined")

# Feature Extraction

In [ ]:
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.color import rgb2gray

def extract_texture_features(img_rgb, mask_bin):
    gray = (rgb2gray(img_rgb) * 255).astype(np.uint8)
    mask_bool = mask_bin.astype(bool)

    if not mask_bool.any():
        return np.zeros(28, dtype=np.float32)

    radius = 3
    n_points = 8 * radius
    lbp = local_binary_pattern(gray, n_points, radius, method='uniform')
    lbp_masked = lbp[mask_bool]
    lbp_hist, _ = np.histogram(lbp_masked, bins=10, range=(0, n_points + 2), density=True)
    lbp_mean = lbp_masked.mean()
    lbp_std = lbp_masked.std()

    gray_masked = gray.copy()
    gray_masked[~mask_bool] = 0

    try:
        glcm = graycomatrix(gray_masked, distances=[1],
                            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                            levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast').mean()
        dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
        homogeneity = graycoprops(glcm, 'homogeneity').mean()
        energy = graycoprops(glcm, 'energy').mean()
        glcm_features = np.array([contrast, dissimilarity, homogeneity, energy])
    except:
        glcm_features = np.zeros(4)

    img_masked = img_rgb[mask_bool]
    rgb_mean = img_masked.mean(axis=0)
    rgb_std = img_masked.std(axis=0)

    img_uint8 = (np.clip(img_rgb, 0, 1) * 255).astype(np.uint8)
    lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
    lab_masked = lab[mask_bool]
    lab_mean = lab_masked.mean(axis=0) / 255.0
    lab_std = lab_masked.std(axis=0) / 255.0

    features = np.concatenate([
        lbp_hist, [lbp_mean, lbp_std], glcm_features,
        rgb_mean, rgb_std, lab_mean, lab_std
    ])

    return features.astype(np.float32)

print("✅ Texture feature extraction defined")

# Model Architecture

In [ ]:
class AttentionCropClassifier(nn.Module):
    def __init__(self, num_classes=4, dropout=0.4):
        super().__init__()

        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.features))

        cnn_feature_dim = 1280
        self.spatial_attention = CBAM_SoftMask_Attention(
            cnn_feature_dim,
            reduction=16,
            leak_weight=0.15
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_feature_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout / 2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes)
        )

    def forward(self, img, mask):
        feats = self.features(img)
        mask_resized = F.interpolate(mask, size=feats.shape[2:], mode='bilinear', align_corners=False)
        attended_feats, attention_map = self.spatial_attention(feats, mask_resized)
        cnn_pooled = self.global_pool(attended_feats).flatten(1)
        logits = self.classifier(cnn_pooled)
        return logits

print("✅ AttentionCropClassifier with CBAM + SoftMask")

# Dataset

In [ ]:
class MultiCropDataset(Dataset):
    def __init__(self, pairs, labels_df, transform=None, min_area=0, padding=20):
        self.transform = transform
        self.min_area = min_area
        self.padding = padding

        self.id_to_label = {}
        for _, row in labels_df.iterrows():
            img_id = os.path.splitext(os.path.basename(row['sample_index']))[0].replace('img_', '')
            self.id_to_label[img_id] = to_idx(row['label'])

        print(f"🔍 Extracting crops from {len(pairs)} images...")
        self.samples = []

        for pair in tqdm(pairs, desc="Extracting crops"):
            img_id = pair['id']
            label = self.id_to_label.get(img_id, 0)

            crops = extract_region_crops(pair['img'], pair['mask'],
                                         min_area=self.min_area,
                                         padding=self.padding,
                                         target_size=(384, 384))

            for crop_idx, crop in enumerate(crops):
                self.samples.append({
                    'img': crop['img'],
                    'mask': crop['mask'],
                    'label': label,
                    'img_id': img_id,
                    'crop_idx': crop_idx
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_tensor = torch.from_numpy(sample['img']).permute(2, 0, 1).float()
        img_tensor = img_tensor / 255.0

        mask_tensor = torch.from_numpy(sample['mask']).unsqueeze(0).float()
        if mask_tensor.max() > 1.0:
            mask_tensor = mask_tensor / 255.0

        label = torch.tensor(sample['label'], dtype=torch.long)

        if self.transform:
            img_tensor = self.transform(img_tensor)

        return img_tensor, mask_tensor, label, sample['img_id']

# Training Setup

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Data augmentation
train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=45),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_pairs, val_pairs = train_test_split(
    train_data,
    test_size=0.2,
    random_state=42,
    stratify=[train_labels[train_labels['sample_index'] == f"img_{p['id']}.png"]['label'].values[0]
              for p in train_data]
)

print(f"📊 Dataset split:")
print(f"   Train images: {len(train_pairs)}")
print(f"   Val images: {len(val_pairs)}")

train_dataset = MultiCropDataset(
    train_pairs,
    train_labels,
    transform=train_transform,
    min_area=0,
    padding=30
)

val_dataset = MultiCropDataset(
    val_pairs,
    train_labels,
    transform=val_transform,
    min_area=0,
    padding=30
)

train_crop_labels = [sample['label'] for sample in train_dataset.samples]
train_counts = pd.Series(train_crop_labels).value_counts().sort_index()
train_counts.index = [CLASS_NAMES[i] for i in train_counts.index]

val_crop_labels = [sample['label'] for sample in val_dataset.samples]
val_counts = pd.Series(val_crop_labels).value_counts().sort_index()
val_counts.index = [CLASS_NAMES[i] for i in val_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(train_counts, labels=train_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title(f'Train Crops (n={len(train_dataset)})')
axes[1].pie(val_counts, labels=val_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Val Crops (n={len(val_dataset)})')
plt.tight_layout()
plt.show()

print(f"\nTrain crops: {len(train_dataset)}")
print(f"Val crops: {len(val_dataset)}")

train_crop_labels = [sample['label'] for sample in train_dataset.samples]
class_counts = np.bincount(train_crop_labels, minlength=4)

print(f"📊 Class distribution in crops:")
for i, count in enumerate(class_counts):
    print(f"   {CLASS_NAMES[i]}: {count} ({count/len(train_dataset)*100:.1f}%)")

class_sample_weights = 1.0 / (class_counts + 1e-6)
class_sample_weights = class_sample_weights / class_sample_weights.min()
sample_weights = [class_sample_weights[label] for label in train_crop_labels]
sample_weights = np.clip(sample_weights, 1, 8)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
    pin_memory=False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(f"\n✅ DataLoaders created (batch_size={BATCH_SIZE})")
print(f"   Train crops: {len(train_dataset)}")
print(f"   Val crops: {len(val_dataset)}")

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        if self.label_smoothing > 0:
            n_classes = inputs.size(1)
            targets_smooth = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0
            )
            targets_smooth = targets_smooth * (1 - self.label_smoothing) + \
                           self.label_smoothing / n_classes
            ce_loss = -(targets_smooth * F.log_softmax(inputs, dim=1)).sum(dim=1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)

        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Training Loop

In [ ]:
# Model
model = AttentionCropClassifier(num_classes=4, dropout=0.6).to(device)
for param in model.features.parameters():
    param.requires_grad = False

print(f"🎯 Model: AttentionCropClassifier")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def compute_class_weights(dataset):
    labels = [sample['label'] for sample in dataset.samples]
    class_counts = np.bincount(labels, minlength=4)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    return torch.tensor(class_weights, dtype=torch.float32)

class_weights = compute_class_weights(train_dataset).to(device)
print(f"📊 Class weights: {class_weights}")

# Loss and optimizer
LEARNING_RATE = 5e-4       # or even 1e-4
WEIGHT_DECAY = 3e-4        # slightly stronger L2
NUM_EPOCHS = 20*3
PATIENCE = 25

num_batches_per_epoch = len(train_loader)

optimizer = Ranger21(
    model.parameters(),
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    num_batches_per_epoch=num_batches_per_epoch,
    weight_decay=WEIGHT_DECAY,
    use_warmup=True,
    use_madgrad=False,
    # optional: turn off some extras
    #use_gc=True,          # Does not exist
    use_adabelief=False,   # can reduce aggressiveness
    #use_normloss=False    # Does not exist
)

# Fix Ranger21 warmdown issues by disabling the problematic warmdown phase
if hasattr(optimizer, 'use_warmdown'):
    optimizer.use_warmdown = False
if hasattr(optimizer, 'warmdown_pct_start'):
    optimizer.warmdown_pct_start = 0.0
if hasattr(optimizer, 'warmdown_min_lr'):
    optimizer.warmdown_min_lr = 0.0

criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)

print(f"\n✅ Training setup complete")
print(f"   Loss: FocalLoss")
print(f"   Optimizer: Ranger21 (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"   Epochs: {NUM_EPOCHS}, Early stopping patience: {PATIENCE}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    num_samples = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='Training')
    for i, (imgs, masks, labels, _) in enumerate(pbar):
        batch_size = labels.size(0)
        imgs = imgs.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs, masks)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if torch.cuda.is_available() and (i % 10 == 0):
            torch.cuda.empty_cache()

        running_loss += loss.item() * batch_size
        num_samples += batch_size

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': loss.item()})

    epoch_loss = running_loss / num_samples
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_f1

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    num_samples = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, masks, labels, _ in tqdm(loader, desc='Validation'):
            batch_size = labels.size(0)
            imgs = imgs.to(device)
            masks = masks.to(device)
            labels = labels.to(device)

            outputs = model(imgs, masks)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * batch_size
            num_samples += batch_size

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / num_samples
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_f1, all_preds, all_labels

print("✅ Training functions defined")

In [ ]:
print("\n📊 Crops per classe:")
for i, name in enumerate(CLASS_NAMES):
    count = sum(1 for s in train_dataset.samples if s['label'] == i)
    print(f"   {name}: {count} crops ({count/len(train_dataset)*100:.1f}%)")

In [ ]:
# TRAINING

history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [], 'lr': []}
best_val_f1 = 0.0
patience_counter = 0

print("\n" + "="*60)
print("🚀 STARTING TRAINING - Attention Crop Classifier")
print("="*60)

for epoch in range(NUM_EPOCHS):
    if epoch == 5:
        for param in model.features.parameters():
            param.requires_grad = True

    print(f"\n{'='*60}")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | LR: {current_lr:.2e}")
    print(f"{'='*60}")

    train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_f1, val_preds, val_labels = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)

    print(f"\n📊 Train Loss: {train_loss:.6f} | Train F1: {train_f1:.6f}")
    print(f"📊    Val Loss: {val_loss:.6f}   |     Val F1: {val_f1:.6f}")

    print("\n📊 Classification Report:")
    print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES, digits=3))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
        }, 'best_attention_crop_model.pth')
        print(f"✅ BEST MODEL! F1: {best_val_f1:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⏳ Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch+1}")
        break

print(f"\n🎉 TRAINING COMPLETE!")
print(f"Best Validation F1: {best_val_f1:.4f}")

# Evaluation & Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o', markersize=4)
axes[0, 0].plot(history['val_loss'], label='Val Loss', marker='s', markersize=4)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['train_f1'], label='Train F1', marker='o', markersize=4)
axes[0, 1].plot(history['val_f1'], label='Val F1', marker='s', markersize=4)
axes[0, 1].axhline(best_val_f1, color='red', linestyle='--', label=f'Best: {best_val_f1:.3f}')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Training & Validation F1')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['lr'], marker='o', markersize=4, color='purple')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_title('Learning Rate Schedule')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

overfit_gaps = [t - v for t, v in zip(history['train_f1'], history['val_f1'])]
axes[1, 1].plot(overfit_gaps, marker='o', markersize=4, color='orange')
axes[1, 1].axhline(0.15, color='red', linestyle='--', label='Warning threshold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Train F1 - Val F1')
axes[1, 1].set_title('Overfitting Gap')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
checkpoint = torch.load('best_attention_crop_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1} (F1: {checkpoint['val_f1']:.4f})")

_, _, val_preds, val_labels = validate(model, val_loader, criterion, device)

cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Validation Confusion Matrix (F1: {best_val_f1:.3f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("\n📊 Classification Report:")
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES, digits=3))

# Test Set Prediction

In [ ]:
def predict_with_tta(model, img, mask, device, n_tta=5):
    model.eval()
    predictions = []

    transforms = [
        lambda x, m: (x, m),
        lambda x, m: (torch.flip(x, [-1]), torch.flip(m, [-1])),
        lambda x, m: (torch.flip(x, [-2]), torch.flip(m, [-2])),
        lambda x, m: (torch.flip(x, [-1, -2]), torch.flip(m, [-1, -2])),
        lambda x, m: (torch.rot90(x, 1, [-2, -1]), torch.rot90(m, 1, [-2, -1])),
    ]

    with torch.no_grad():
        for transform in transforms[:n_tta]:
            img_aug, mask_aug = transform(img, mask)
            output = model(img_aug, mask_aug)
            predictions.append(F.softmax(output, dim=1))

    return torch.stack(predictions).mean(dim=0)

In [ ]:
test_labels_dummy = pd.DataFrame({
    'sample_index': [f"img_{p['id']}.png" for p in test_data],
    'label': ['Luminal A'] * len(test_data)
})

test_dataset = MultiCropDataset(
    test_data,
    test_labels_dummy,
    transform=val_transform,
    min_area=0,
    padding=30
)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

checkpoint = torch.load('best_attention_crop_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
crop_predictions = []
crop_img_ids = []

print("🔮 Predicting on test set...")
with torch.no_grad():
    for imgs, masks, _, img_ids in tqdm(test_loader, desc='Prediction'):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, masks)
        preds = outputs.argmax(dim=1)
        crop_predictions.extend(preds.cpu().numpy())
        crop_img_ids.extend(img_ids)

from collections import Counter

img_predictions = {}
with torch.no_grad():
    for imgs, masks, _, img_ids in tqdm(test_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, masks)
        probs = F.softmax(outputs, dim=1)
        for img_id, prob in zip(img_ids, probs.cpu().numpy()):
            if img_id not in img_predictions:
                img_predictions[img_id] = []
            img_predictions[img_id].append(prob)

final_predictions = {}
for img_id, prob_list in img_predictions.items():
    avg_prob = np.mean(prob_list, axis=0)
    final_pred = avg_prob.argmax()
    final_predictions[img_id] = to_name(final_pred)

sample_ids = [f"img_{img_id}.png" for img_id in final_predictions.keys()]
predictions = list(final_predictions.values())

results_df = pd.DataFrame({
    'sample_index': sample_ids,
    'label': predictions
})

results_df.to_csv('predictions_attention_crop-v5.csv', index=False)

print("✅ Predictions saved to predictions_attention_crop-v5.csv")
print(f"\n📊 Total crops processed: {len(crop_predictions)}")
print(f"📊 Unique images: {len(final_predictions)}")
print(f"📊 Average crops per image: {len(crop_predictions) / len(final_predictions):.2f}")
print("\n📊 Prediction distribution:")
print(results_df['label'].value_counts())

In [ ]:
class_names = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']


In [ ]:
import copy
from torch.utils.data import ConcatDataset, Dataset

CONF_THRESHOLD = 0.7
pseudo_samples = []

print(f"🕵️‍♂️ Mining Pseudo-Labels (Threshold: {CONF_THRESHOLD})...")

model.eval()
dataset_idx = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Pseudo-Labeling"):
        if len(batch) == 4:
            imgs, masks, _, img_ids = batch
        else:
            imgs, masks, img_ids = batch

        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, mask=masks)
        probs = F.softmax(outputs, dim=1)
        max_probs, preds = torch.max(probs, dim=1)
        max_probs = max_probs.cpu().numpy()
        preds = preds.cpu().numpy()

        batch_size = imgs.size(0)
        for b in range(batch_size):
            if max_probs[b] >= CONF_THRESHOLD:
                raw_sample = copy.deepcopy(test_dataset.samples[dataset_idx])
                raw_sample['label'] = int(preds[b])
                pseudo_samples.append(raw_sample)
            dataset_idx += 1

print(f"\n✅ Found {len(pseudo_samples)} confident samples out of {len(test_dataset)}.")
if len(pseudo_samples) > 0:
    pseudo_df = pd.DataFrame(pseudo_samples)
    print("Distribution of new knowledge:")
    print(pseudo_df['label'].value_counts().rename(index={i:n for i,n in enumerate(class_names)}))
else:
    print("⚠️ No samples met the threshold. Try lowering it.")

In [ ]:
class PseudoDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]['img'], self.samples[idx]['label']

In [ ]:
class SimpleCropDataset(Dataset):
    def __init__(self, samples_list, transform=None):
        self.samples = samples_list
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_tensor = torch.from_numpy(sample['img']).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(sample['mask']).unsqueeze(0).float()
        if mask_tensor.max() > 1.0: mask_tensor /= 255.0
        label = torch.tensor(sample['label'], dtype=torch.long)
        if self.transform:
            img_tensor = self.transform(img_tensor)
        return img_tensor, mask_tensor, label, sample.get('img_id', 'pseudo')

if len(pseudo_samples) > 0:
    all_samples = train_dataset.samples + pseudo_samples
    extended_dataset = SimpleCropDataset(all_samples, transform=train_transform)
    print(f"📚 Dataset grew from {len(train_dataset)} to {len(extended_dataset)} samples.")
else:
    extended_dataset = train_dataset

In [ ]:
if len(pseudo_samples) > 0:
    all_labels = [s['label'] for s in extended_dataset.samples]
    counts = pd.Series(all_labels).value_counts().sort_index()
    print("New Class Counts:", counts.values)

    weights = 1.0 / counts
    sample_weights = [weights[t] for t in all_labels]

    new_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    extended_loader = DataLoader(
        extended_dataset,
        batch_size=BATCH_SIZE,
        sampler=new_sampler,
        num_workers=2
    )

    for param in model.features.parameters():
        param.requires_grad = True

    best_pseudo_f1 = 0.0

    for epoch in range(5):
        train_loss, train_f1 = train_epoch(model, extended_loader, criterion, optimizer, device)
        val_loss, val_f1, val_preds, val_labels = validate(model, val_loader, criterion, device)
        print(f"Pseudo-Epoch {epoch+1} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")
        if val_f1 > best_pseudo_f1:
            best_pseudo_f1 = val_f1
            torch.save(model.state_dict(), 'best_model_pseudo.pth')
            print(f"   💾 Prediction Reinforced! New Best F1: {best_pseudo_f1:.4f}")

In [ ]:
print("🔄 Loading Self-Trained Model (best_model_pseudo.pth)...")
try:
    checkpoint = torch.load('best_model_pseudo.pth')
    model.load_state_dict(checkpoint)
    model.eval()
    print("✅ Model loaded successfully!")
except FileNotFoundError:
    print("❌ Error: 'best_model_pseudo.pth' not found. Did Stage 3 finish?")

model.eval()
crop_predictions = []
crop_img_ids = []

print("🔮 Predicting on test set (self-trained)...")
with torch.no_grad():
    for imgs, masks, _, img_ids in tqdm(test_loader, desc='Prediction'):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, masks)
        preds = outputs.argmax(dim=1)
        crop_predictions.extend(preds.cpu().numpy())
        crop_img_ids.extend(img_ids)

img_predictions = {}
with torch.no_grad():
    for imgs, masks, _, img_ids in tqdm(test_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, masks)
        probs = F.softmax(outputs, dim=1)
        for img_id, prob in zip(img_ids, probs.cpu().numpy()):
            if img_id not in img_predictions:
                img_predictions[img_id] = []
            img_predictions[img_id].append(prob)

final_predictions = {}
for img_id, prob_list in img_predictions.items():
    avg_prob = np.mean(prob_list, axis=0)
    final_pred = avg_prob.argmax()
    final_predictions[img_id] = to_name(final_pred)

sample_ids = [f"img_{img_id}.png" for img_id in final_predictions.keys()]
predictions = list(final_predictions.values())

results_df = pd.DataFrame({
    'sample_index': sample_ids,
    'label': predictions
})

results_df.to_csv('predictions_attention_crop_pseudo.csv', index=False)

print("✅ Predictions saved to predictions_attention_crop_pseudo.csv")
print(f"\n📊 Total crops processed: {len(crop_predictions)}")
print(f"📊 Unique images: {len(final_predictions)}")
print(f"📊 Average crops per image: {len(crop_predictions) / len(final_predictions):.2f}")
print("\n📊 Prediction distribution:")
print(results_df['label'].value_counts())

In [ ]:
import copy
from torch.utils.data import ConcatDataset, Dataset

CONF_THRESHOLD = 0.4
pseudo_samples = []

print(f"🕵️‍♂️ Mining Pseudo-Labels (Threshold: {CONF_THRESHOLD})...")

model.eval()
dataset_idx = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Pseudo-Labeling"):
        if len(batch) == 4:
            imgs, masks, _, img_ids = batch
        else:
            imgs, masks, img_ids = batch

        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, mask=masks)
        probs = F.softmax(outputs, dim=1)
        max_probs, preds = torch.max(probs, dim=1)
        max_probs = max_probs.cpu().numpy()
        preds = preds.cpu().numpy()

        batch_size = imgs.size(0)
        for b in range(batch_size):
            if max_probs[b] >= CONF_THRESHOLD:
                raw_sample = copy.deepcopy(test_dataset.samples[dataset_idx])
                raw_sample['label'] = int(preds[b])
                pseudo_samples.append(raw_sample)
            dataset_idx += 1

print(f"\n✅ Found {len(pseudo_samples)} confident samples out of {len(test_dataset)}.")
if len(pseudo_samples) > 0:
    pseudo_df = pd.DataFrame(pseudo_samples)
    print("Distribution of new knowledge:")
    print(pseudo_df['label'].value_counts().rename(index={i:n for i,n in enumerate(class_names)}))
else:
    print("⚠️ No samples met the threshold. Try lowering it.")

In [ ]:
class PseudoDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]['img'], self.samples[idx]['label']

In [ ]:
class SimpleCropDataset(Dataset):
    def __init__(self, samples_list, transform=None):
        self.samples = samples_list
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_tensor = torch.from_numpy(sample['img']).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(sample['mask']).unsqueeze(0).float()
        if mask_tensor.max() > 1.0: mask_tensor /= 255.0
        label = torch.tensor(sample['label'], dtype=torch.long)
        if self.transform:
            img_tensor = self.transform(img_tensor)
        return img_tensor, mask_tensor, label, sample.get('img_id', 'pseudo')

if len(pseudo_samples) > 0:
    all_samples = train_dataset.samples + pseudo_samples
    extended_dataset = SimpleCropDataset(all_samples, transform=train_transform)
    print(f"📚 Dataset grew from {len(train_dataset)} to {len(extended_dataset)} samples.")
else:
    extended_dataset = train_dataset

In [ ]:

if len(pseudo_samples) > 0:
    # 1. Recalculate Class Counts for the NEW dataset
    all_labels = [s['label'] for s in extended_dataset.samples]
    counts = pd.Series(all_labels).value_counts().sort_index()
    print("New Class Counts:", counts.values)

    # Initialize Optimizer
    optimizer = Ranger21(
        model.parameters(),
        lr=LEARNING_RATE,
        num_epochs=10,
        num_batches_per_epoch=num_batches_per_epoch,
        weight_decay=WEIGHT_DECAY,  # Regularization to fight overfitting
        use_warmup=True,            # Smooth start
        use_madgrad=False, # Toggle this to True# Standard Ranger21 core
    )

    # 2. Calculate Weights (Inverse Frequency)
    weights = 1.0 / counts
    sample_weights = [weights[t] for t in all_labels]

    # 3. New Sampler
    new_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    # 4. New Loader
    extended_loader = DataLoader(
        extended_dataset,
        batch_size=BATCH_SIZE,
        sampler=new_sampler,
        num_workers=2
    )


    for param in model.features.parameters():
      param.requires_grad = True

    best_pseudo_f1 = 0.0

    for epoch in range(10):
      # Training
      train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)

      # Validation
      val_loss, val_f1, val_preds, val_labels = validate(model, val_loader, criterion, device)

      print(f"Pseudo-Epoch {epoch+1} | Train F1: {train_f1} |Val F1: {val_f1:.4f}")

        # Save if it improves
      if val_f1 > best_pseudo_f1:
          best_pseudo_f1 = val_f1
          torch.save(model.state_dict(), 'best_model_pseudo.pth')
          print(f"   💾 Prediction Reinforced! New Best F1: {best_pseudo_f1:.4f}")



In [ ]:
checkpoint = torch.load('best_model_pseudo.pth')
model.load_state_dict(checkpoint)
val_f1 = best_pseudo_f1
#odel.eval()
print(f"✅ Loaded best model (F1: {val_f1:.4f})")

# Re-evaluate on validation
_, _, val_preds, val_labels = validate(model, val_loader, criterion, device)

cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Validation Confusion Matrix (F1: {best_val_f1:.3f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Per-class F1
print("\n📊 Classification Report:")
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES, digits=3))

In [ ]:
print("🔄 Loading Self-Trained Model (best_model_pseudo.pth)...")
try:
    checkpoint = torch.load('best_model_pseudo.pth')
    model.load_state_dict(checkpoint)
    model.eval()
    print("✅ Model loaded successfully!")
except FileNotFoundError:
    print("❌ Error: 'best_model_pseudo.pth' not found. Did Stage 3 finish?")

model.eval()
crop_predictions = []
crop_img_ids = []

print("🔮 Predicting on test set (self-trained)...")
with torch.no_grad():
    for imgs, masks, _, img_ids in tqdm(test_loader, desc='Prediction'):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, masks)
        preds = outputs.argmax(dim=1)
        crop_predictions.extend(preds.cpu().numpy())
        crop_img_ids.extend(img_ids)

img_predictions = {}
with torch.no_grad():
    for imgs, masks, _, img_ids in tqdm(test_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs, masks)
        probs = F.softmax(outputs, dim=1)
        for img_id, prob in zip(img_ids, probs.cpu().numpy()):
            if img_id not in img_predictions:
                img_predictions[img_id] = []
            img_predictions[img_id].append(prob)

final_predictions = {}
for img_id, prob_list in img_predictions.items():
    avg_prob = np.mean(prob_list, axis=0)
    final_pred = avg_prob.argmax()
    final_predictions[img_id] = to_name(final_pred)

sample_ids = [f"img_{img_id}.png" for img_id in final_predictions.keys()]
predictions = list(final_predictions.values())

results_df = pd.DataFrame({
    'sample_index': sample_ids,
    'label': predictions
})

results_df.to_csv('/gdrive/MyDrive/AN2DL_Chall2/predictions_attention_crop_pseudo_04-2-12313.csv', index=False)

print("✅ Predictions saved to predictions_attention_crop_pseudo.csv")
print(f"\n📊 Total crops processed: {len(crop_predictions)}")
print(f"📊 Unique images: {len(final_predictions)}")
print(f"📊 Average crops per image: {len(crop_predictions) / len(final_predictions):.2f}")
print("\n📊 Prediction distribution:")
print(results_df['label'].value_counts())